In [115]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [116]:
df = pd.read_csv("/content/student-por.csv")

In [117]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [118]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      649 non-null    object
 1   sex         649 non-null    object
 2   age         649 non-null    int64 
 3   address     649 non-null    object
 4   famsize     649 non-null    object
 5   Pstatus     649 non-null    object
 6   Medu        649 non-null    int64 
 7   Fedu        649 non-null    int64 
 8   Mjob        649 non-null    object
 9   Fjob        649 non-null    object
 10  reason      649 non-null    object
 11  guardian    649 non-null    object
 12  traveltime  649 non-null    int64 
 13  studytime   649 non-null    int64 
 14  failures    649 non-null    int64 
 15  schoolsup   649 non-null    object
 16  famsup      649 non-null    object
 17  paid        649 non-null    object
 18  activities  649 non-null    object
 19  nursery     649 non-null    object
 20  higher    

In [119]:
df.drop(columns=['school','address','famsize','reason','traveltime'],inplace=True)

In [120]:
df.head()

,sex,age,Pstatus,Medu,Fedu,Mjob,Fjob,guardian,studytime,failures,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,F,18,A,4,4,at_home,teacher,mother,2,0,...,4,3,4,1,1,3,4,0,11,11
1,F,17,T,1,1,at_home,other,father,2,0,...,5,3,3,1,1,3,2,9,11,11
2,F,15,T,1,1,at_home,other,mother,2,0,...,4,3,2,2,3,3,6,12,13,12
3,F,15,T,4,2,health,services,mother,3,0,...,3,2,2,1,1,5,0,14,14,14
4,F,16,T,3,3,other,other,father,2,0,...,4,3,2,1,2,5,0,11,13,13


In [121]:
df['Mjob'].nunique()

5

In [122]:
df['Fjob'].nunique()

5

In [123]:
X = df.drop(columns=['G1','G2','G3'])
y = df['G3']

In [124]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

In [125]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 28 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   sex         649 non-null    object
 1   age         649 non-null    int64 
 2   Pstatus     649 non-null    object
 3   Medu        649 non-null    int64 
 4   Fedu        649 non-null    int64 
 5   Mjob        649 non-null    object
 6   Fjob        649 non-null    object
 7   guardian    649 non-null    object
 8   studytime   649 non-null    int64 
 9   failures    649 non-null    int64 
 10  schoolsup   649 non-null    object
 11  famsup      649 non-null    object
 12  paid        649 non-null    object
 13  activities  649 non-null    object
 14  nursery     649 non-null    object
 15  higher      649 non-null    object
 16  internet    649 non-null    object
 17  romantic    649 non-null    object
 18  famrel      649 non-null    int64 
 19  freetime    649 non-null    int64 
 20  goout     

In [126]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(handle_unknown='ignore')
X_enc =  enc.fit_transform(X[['sex','Pstatus','Mjob','Fjob','guardian','schoolsup','famsup','paid','activities','nursery','higher','internet','romantic']])

In [127]:
from scipy.sparse import hstack

X_final = hstack([X[num_cols].values, X_enc])

In [128]:
X_final = X_final.toarray()

In [129]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

In [130]:
model = Sequential()

model.add(Dense(80, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(60, activation='relu'))
model.add(Dense(1, activation='linear'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [131]:
history_no_dropout = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    verbose=False
)

## Dropout after final layer (0.2)

In [132]:
model = Sequential()

model.add(Dense(80, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(60, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='linear'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
history_dropout_02 = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    verbose=False
)

## Dropout after final layer (0.5)

In [133]:
model = Sequential()

model.add(Dense(80, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(60, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='linear'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
history_dropout_05 = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    verbose=False
)

## Dropout after each layer (0.2)

In [134]:
model = Sequential()

model.add(Dense(80, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.2))
model.add(Dense(60, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='linear'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
history_dropout_each = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    verbose=False
)

In [135]:
results = pd.DataFrame({
    'Model': [
        'No Dropout',
        'Dropout 0.2',
        'Dropout 0.5',
        'Dropout Each Layer 0.2'
    ],
    'Final Train Loss': [
        history_no_dropout.history['loss'][-1],
        history_dropout_02.history['loss'][-1],
        history_dropout_05.history['loss'][-1],
        history_dropout_each.history['loss'][-1]
    ],
    'Final Val Loss': [
        history_no_dropout.history['val_loss'][-1],
        history_dropout_02.history['val_loss'][-1],
        history_dropout_05.history['val_loss'][-1],
        history_dropout_each.history['val_loss'][-1]
    ],
    'Final Train MAE': [
        history_no_dropout.history['mae'][-1],
        history_dropout_02.history['mae'][-1],
        history_dropout_05.history['mae'][-1],
        history_dropout_each.history['mae'][-1]
    ],
    'Final Val MAE': [
        history_no_dropout.history['val_mae'][-1],
        history_dropout_02.history['val_mae'][-1],
        history_dropout_05.history['val_mae'][-1],
        history_dropout_each.history['val_mae'][-1]
    ]
})

results

,Model,Final Train Loss,Final Val Loss,Final Train MAE,Final Val MAE
0,No Dropout,3.446158,9.701571,1.385019,2.401280
1,Dropout 0.2,5.515029,9.927496,1.791280,2.374594
2,Dropout 0.5,10.590089,8.900297,2.542192,2.284102
3,Dropout Each Layer 0.2,6.974318,9.350545,2.022114,2.368804
